In [8]:
import sys
sys.path.append(r"/")

from DATA.runner import run_valuation_range, run_valuation_for_tickers

# 인덱스 범위 실행
res = run_valuation_range(start=0, end=3, batch_size=20)
print(res)
# -> {'success_tickers': ..., 'valuation_rows_upserted': ..., 'rev_fc_rows_upserted': ...}

# # 2) 티커 리스트를 직접 넣어 실행
# result2 = run_valuation_for_tickers(
#     tickers=["AAPL","MSFT","META"],
#     batch_size=10,
#     measurement_date="2025-10-13",  # 같은 날은 갱신, 다른 날은 누적
# )
# print(result2)

[TICKERS] count=3
[AUDIT] US_fundq missing: 0 tickers
[]
[AUDIT] US_fundm missing: 1 tickers
['MSFT']
[AUDIT-SAVE] US_fundm missing 1 tickers saved -> market_cap_missing_ticker_20251013.csv
[TICKER] 1/3 AAPL
[EXC-TICKER] AAPL e=module 'DATA.stock_invest_function' has no attribute 'fetch_revenue_data' tb=AttributeError: module 'DATA.stock_invest_function' has no attribute 'fetch_revenue_data'
[ERR] {'ticker': 'AAPL', 'stage': 'pipeline', 'error': "module 'DATA.stock_invest_function' has no attribute 'fetch_revenue_data'"}
[TICKER] 2/3 MSFT
[EXC-TICKER] MSFT e=module 'DATA.stock_invest_function' has no attribute 'fetch_revenue_data' tb=AttributeError: module 'DATA.stock_invest_function' has no attribute 'fetch_revenue_data'
[ERR] {'ticker': 'MSFT', 'stage': 'pipeline', 'error': "module 'DATA.stock_invest_function' has no attribute 'fetch_revenue_data'"}
[TICKER] 3/3 AMZN
[EXC-TICKER] AMZN e=module 'DATA.stock_invest_function' has no attribute 'fetch_revenue_data' tb=AttributeError: modul

In [1]:
# import sys, os
# from pathlib import Path
# import pandas as pd
# import numpy as np
# from typing import Optional
# # from stock_forecast.Korea_Market.valuation.kse_valuation_machine_v1 import psr_forecast_df
#
#
# def add_repo_path():
#     here = Path.cwd()
#     # 현재 위치부터 상위 폴더를 훑으며 DATA 폴더가 보이는 지점 찾기
#     for p in [here, *here.parents]:
#         if (p / "DATA").exists():
#             if str(p) not in sys.path:
#                 sys.path.insert(0, str(p))
#             return str(p)
#     # 못 찾으면 로컬 고정 경로(본인 PC 경로로) 마지막 보루로 추가
#     fallback = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast"
#     if os.path.isdir(fallback) and fallback not in sys.path:
#         sys.path.insert(0, fallback)
#     return fallback
#
# project_path = add_repo_path()
# print("Using project path:", project_path)
#
# import calendar
# import time
# # from DATA.stock_invest_function import *
# from DATA.stock_invest_function import *
# from datetime import datetime
# from dateutil.relativedelta import relativedelta
# warnings.filterwarnings('ignore')

Using project path: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


In [2]:
# # 유틸리티 함수들
# def convert_to_month_end(date_str):
#     try:
#         # 문자열/타입 혼용 안전 변환
#         date_obj = pd.to_datetime(date_str)
#         if pd.isna(date_obj):
#             return None
#
#         y, m, d = date_obj.year, date_obj.month, date_obj.day
#
#         # 1~5일 → 전달 말일
#         if 1 <= d <= 5:
#             if m == 1:
#                 prev_y, prev_m = y - 1, 12
#             else:
#                 prev_y, prev_m = y, m - 1
#             last_day_prev = calendar.monthrange(prev_y, prev_m)[1]
#             return datetime(prev_y, prev_m, last_day_prev)
#
#         # 그 외 → 해당월 말일
#         last_day_cur = calendar.monthrange(y, m)[1]
#         return datetime(y, m, last_day_cur)
#
#     except Exception:
#         return None
#
# def process_daily_to_monthly_market_data(daily_data, ticker):
#     if not daily_data:
#         return pd.DataFrame()
#     df = pd.DataFrame(daily_data)
#     df['date'] = pd.to_datetime(df['date'])
#     df = df.sort_values('date')
#     df['year_month'] = df['date'].dt.to_period('M')
#     monthly_data = []
#     for year_month in df['year_month'].unique():
#         month_data = df[df['year_month'] == year_month]
#         last_day_data = month_data.loc[month_data['date'].idxmax()]
#         monthly_data.append({
#             'ticker': ticker,
#             'date': last_day_data['date'],
#             'market_cap': last_day_data['marketCap'],
#             'market_cap_billions': round(last_day_data['marketCap'] / 1_000_000_000, 2),
#         })
#     return pd.DataFrame(monthly_data)
#
#
# def fetch_revenue_data(ticker, api_key):
#     url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
#     params = {'limit': 200, 'apikey': api_key, 'period': 'quarter'}
#     try:
#         response = requests.get(url, params=params, timeout=30)
#         if response.status_code != 200:
#             return None, f"HTTP {response.status_code}"
#         data = response.json()
#         if isinstance(data, dict) and 'Error Message' in data:
#             return None, f"API 오류: {data['Error Message']}"
#         if not data:
#             return None, "데이터 없음"
#         return data, None
#     except Exception as e:
#         return None, f"오류: {str(e)}"
#
# def fetch_market_data_yearly(ticker, api_key, start_year=2010):
#     all_data = []
#     current_year = datetime.now().year
#     for year in range(start_year, current_year + 1):
#         start_date_str = f"{year}-01-01"
#         end_date_str = f"{year}-12-31"
#         url = f"https://financialmodelingprep.com/api/v3/historical-market-capitalization/{ticker}"
#         params = {'from': start_date_str, 'to': end_date_str, 'apikey': api_key}
#         try:
#             response = requests.get(url, params=params, timeout=30)
#             if response.status_code == 200:
#                 data = response.json()
#                 if data and isinstance(data, list):
#                     all_data.extend(data)
#             time.sleep(0.3)
#         except Exception as e:
#             continue
#     return all_data if all_data else None, None
#
# def fetch_db_revenue_data(ticker, db_info, end_date='2025-08-31'):
#     try:
#         engine = create_engine(
#             f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
#             f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
#         )
#         query = f"""
#         SELECT date, ticker, saleq
#         FROM US_fundq
#         WHERE ticker = '{ticker}'
#         AND saleq IS NOT NULL
#         AND date <= '{end_date}'
#         ORDER BY date ASC
#         """
#         df = pd.read_sql(query, con=engine)
#         engine.dispose()
#         if not df.empty:
#             df['date'] = pd.to_datetime(df['date'])
#             df['revenue_billions'] = df['saleq'] / 1000
#             df['date_month_end'] = df['date'].apply(convert_to_month_end)
#         return df[['ticker', 'date', 'date_month_end', 'revenue_billions']]
#     except Exception as e:
#         return pd.DataFrame()
#
# def fetch_db_market_data(ticker, db_info, end_date='2024-12-31'):
#     try:
#         engine = create_engine(
#             f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
#             f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
#         )
#         query = f"""
#         SELECT date, ticker, me
#         FROM US_fundm
#         WHERE ticker = '{ticker}'
#         AND me IS NOT NULL
#         AND date <= '{end_date}'
#         ORDER BY date ASC
#         """
#         df = pd.read_sql(query, con=engine)
#         engine.dispose()
#         if not df.empty:
#             df['date'] = pd.to_datetime(df['date'])
#             df['market_cap_billions'] = df['me'] / 1000
#             df['date_month_end'] = df['date'].apply(convert_to_month_end)
#         return df[['ticker', 'date', 'date_month_end', 'market_cap_billions']]
#     except Exception as e:
#         return pd.DataFrame()
#
# def calculate_enhanced_ttm_and_psr(merged_data):
#     """Calculate enhanced TTM and PSR"""
#     df = merged_data.copy()
#
#     # ✅ 날짜형으로 변환 (핵심 수정)
#     df['date_month_end'] = pd.to_datetime(df['date_month_end'], errors='coerce')
#     df = df.sort_values(['date_month_end']).reset_index(drop=True)
#     df = df.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)
#
#     # Calculate TTM from quarterly revenue
#     df['revenue_ttm'] = df.groupby('ticker')['revenue_billions'].rolling(window=4, min_periods=1).sum().reset_index(0,
#                                                                                                                     drop=True)
#     df['revenue_ttm_billions'] = df['revenue_ttm']
#
#     # Apply 2-month shift
#     df['revenue_ttm_shift'] = df.groupby('ticker')['revenue_ttm_billions'].shift(2)
#
#     # Calculate PSR
#     df['PSR_ttm'] = df['market_cap_billions'] / df['revenue_ttm_shift']
#
#     # Handle infinite values
#     df['PSR_ttm'] = df['PSR_ttm'].replace([np.inf, -np.inf], np.nan)
#
#     return df
#
# def prepare_revenue_ttm(
#     df: pd.DataFrame,
#     revenue_key: str = "revenue_billions",
#     min_periods: int = 1,   # 완전한 TTM만 원하면 4로 바꾸세요
# ) -> pd.DataFrame:
#     """
#     1) revenue 칼럼들의 NaN을 '해당 행의 revenue 평균'으로 채움
#     2) 각 revenue 칼럼의 4분기 합(TTM)을 *_ttm 칼럼으로 생성 (시차 없음)
#     - 그룹 기준: ticker
#     - 정렬 기준: date_month_end (월말 날짜)
#     """
#     d = df.copy()
#
#     # --- 키 정리 ---
#     # date_month_end: index에 있으면 칼럼으로 복구
#     if 'date_month_end' not in d.columns:
#         d = d.reset_index().rename(columns={'index': 'date_month_end'})
#     d['date_month_end'] = pd.to_datetime(d['date_month_end'])
#
#     if 'ticker' not in d.columns:
#         raise ValueError("ticker 칼럼이 필요합니다.")
#
#     # --- revenue 칼럼 자동 탐지 ---
#     rev_cols = [c for c in d.columns if revenue_key in c]
#     if not rev_cols:
#         raise ValueError(f"'{revenue_key}' 가 포함된 칼럼을 찾지 못했습니다.")
#
#     # --- ticker NaN 보정 ---
#     # 단일 티커면 ffill/bfill로 채움, 복수 티커면 NaN 행 제거(필요 시 정책 조정)
#     uniq_tickers = d['ticker'].dropna().unique()
#     if len(uniq_tickers) == 1:
#         d['ticker'] = d['ticker'].ffill().bfill()
#     else:
#         d = d[~d['ticker'].isna()].copy()
#
#     # --- 정렬 ---
#     d = d.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)
#
#     # --- NaN 보간: 행 단위 평균으로 revenue 결측치 채우기 ---
#     row_mean = d[rev_cols].mean(axis=1, skipna=True)
#     for c in rev_cols:
#         d[c] = d[c].fillna(row_mean)
#
#     # --- TTM 계산 (최근 4분기 합, 시차 없음) ---
#     for c in rev_cols:
#         ttm_col = f"{c}_ttm"
#         d[ttm_col] = (
#             d.groupby('ticker', group_keys=False)[c]
#              .rolling(window=4, min_periods=min_periods)
#              .sum()
#              .reset_index(level=0, drop=True)
#         )
#
#     d = d.set_index('date_month_end')
#
#     return d
#
# def clean_rev_data(rev_data: pd.DataFrame) -> pd.DataFrame:
#     """
#     1) 'revenue' 컬럼 값이 NaN인 행 제거
#     2) (calendar_year, period) 중복 행 제거 (첫 번째 행만 유지)
#        - 입력 순서를 그대로 기준으로 '첫째 데이터'를 보존
#     """
#     required = ['revenue', 'calendar_year', 'period']
#     missing = [c for c in required if c not in rev_data.columns]
#     if missing:
#         raise ValueError(f"필수 컬럼이 없습니다: {missing}")
#
#     d = rev_data.copy()
#
#     # 1) revenue NaN인 행 제거
#     before = len(d)
#     d = d[~d['revenue'].isna()].copy()
#     removed_nan = before - len(d)
#
#     # 2) (calendar_year, period) 중복 제거 — 첫 행 유지(현재 순서 기준)
#     before2 = len(d)
#     d = d.drop_duplicates(subset=['calendar_year', 'period'], keep='first').reset_index(drop=True)
#     removed_dup = before2 - len(d)
#
#     print(f"[clean_rev_data_minimal] removed rows → revenue NaN: {removed_nan}, duplicates: {removed_dup}")
#     return d
#
# # ==========================
# # 사용 예시
# # ==========================
# # cleaned = clean_rev_data(rev_data)
# # cleaned.head()


In [3]:
# 설정값들
ticker = 'MU'

hs_code = '841191'

api_key = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    # 'host': '192.168.0.230',
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

start_date_month = '2011-03-01'
end_date_month = (pd.Timestamp.today().normalize() - pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')

In [4]:
print("=" * 80)
print("전처리 과정 테스트 시작")
print(f"대상 종목: {ticker}")
print("=" * 80)

# 1. FMP 매출 데이터 수집
print("\n1. FMP 매출 데이터 수집 중...")
revenue_data, error = fetch_revenue_data(ticker, api_key)

if revenue_data is None:
    print(f"ERROR: FMP 매출 데이터 수집 실패 - {error}")
    exit()

all_revenue_data = []
for item in revenue_data:
    all_revenue_data.append({
        'ticker': ticker,
        'date': item.get('date', ''),
        'calendar_year': item.get('calendarYear', ''),
        'period': item.get('period', ''),
        'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
        'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
    })

fmp_revenue_df = pd.DataFrame(all_revenue_data)
fmp_revenue_df['date'] = pd.to_datetime(fmp_revenue_df['date'])
fmp_revenue_df = fmp_revenue_df.sort_values(['ticker', 'date'])
fmp_revenue_df['date_month_end'] = fmp_revenue_df['date'].apply(convert_to_month_end)
fmp_revenue_df = fmp_revenue_df.drop_duplicates(subset=['date_month_end'], keep='first').reset_index(drop=True)
print(f"FMP 매출 데이터: {len(fmp_revenue_df)}건")


# 2. DB 매출 데이터 가져오기
db_revenue_raw = fetch_db_revenue_data(ticker, db_info)
db_revenue_df = db_revenue_raw.loc[db_revenue_raw['revenue_billions'] != db_revenue_raw['revenue_billions'].shift()]

# db_revenue_df = db_revenue_raw.drop_duplicates(subset=['date_month_end', 'revenue_billions'], keep='first')
mereged_rev_data = pd.merge(fmp_revenue_df, db_revenue_df, on = ['ticker', 'date_month_end'], how='outer')

rev_data = mereged_rev_data[mereged_rev_data['date_month_end'] >= start_date_month ]
rev_data['revenue_billions_x'] = rev_data['revenue_billions_x'].fillna(rev_data['revenue_billions_y'])
# 컬럼 이름 변경
rev_data.rename(columns={'revenue_billions_x': 'revenue_billions'}, inplace=True)

rev_data = clean_rev_data(rev_data)

전처리 과정 테스트 시작
대상 종목: MU

1. FMP 매출 데이터 수집 중...
FMP 매출 데이터: 161건
[clean_rev_data_minimal] removed rows → revenue NaN: 0, duplicates: 0


In [5]:
import importlib
import DATA.us_sarima_forecast as sarima
importlib.reload(sarima)
import DATA.us_lstm_forecast_v2 as lstm_v2
importlib.reload(lstm_v2)
import DATA.us_prophet_forecast_v3 as prophet_v3
importlib.reload(prophet_v3)
import DATA.us_est_forecast_v2 as esmod
importlib.reload(esmod)

<module 'DATA.us_est_forecast_v2' from 'C:\\Users\\82108\\OneDrive\\바탕 화면\\investment\\investment_strategy\\DATA\\us_est_forecast_v2.py'>

In [6]:
# periods=4 또는 8 등 원하는 분기 수
periods = 4

# 모듈 함수 호출 (내부에서 월말 정렬/중복제거 처리)
sarima_df, results = sarima.run_sarima_prediction(
    rev_data,
    forecast_quarters=periods,   # ← 예측 분기 수
    exog_col=None                # 외생변수 없으면 None
)

# 인덱스를 date_month_end로 설정
sarima_df = sarima_df.sort_values("date_month_end").set_index("date_month_end")

In [7]:
# 1) 4분기 예측
# 4분기 예측
lstm_raw_df, lstm_results_4q = lstm_v2.run_lstm_revenue_prediction(rev_data, ticker=ticker, prediction_quarters=4)
lstm_df = lstm_raw_df.drop_duplicates(subset=['revenue_billions_lstm_forecast'], keep='last')

In [8]:
# 4분기 예측
prophet_raw_df, res_4q = prophet_v3.run_prophet_revenue_only(rev_data, ticker=ticker, prediction_quarters=4)

15:29:42 - cmdstanpy - INFO - Chain [1] start processing
15:29:42 - cmdstanpy - INFO - Chain [1] done processing


In [9]:
# 4분기 예측
es_raw_df, res_q4 = esmod.run_es_revenue_quarterly(rev_data, ticker=ticker, prediction_quarters=4)
# es_raw_df.tail(24)


In [10]:
es_raw_df.tail(10)

,revenue_billions,revenue_billions_esq_forecast
date_month_end,,
2024-05-31,6.81,6.810000
2024-08-31,7.75,7.750000
2024-11-30,8.71,8.710000
2025-02-28,8.05,8.050000
2025-05-31,9.30,9.300000
2025-08-31,11.31,11.310000
2025-11-30,NaN,12.812249
2026-02-28,NaN,14.314498
2026-05-31,NaN,15.816748


In [11]:
# 3. FMP 시가총액 데이터 수집
print("2. FMP 시가총액 데이터 수집 중...")
market_data, error = fetch_market_data_yearly(ticker, api_key, start_year=2010)

if not market_data:
    print("ERROR: FMP 시가총액 데이터 수집 실패")
    raise SystemExit(1)

fmp_market_df = process_daily_to_monthly_market_data(market_data, ticker).copy()
fmp_market_df['date_month_end'] = fmp_market_df['date'].apply(convert_to_month_end)
# 혹시 중복/정렬 문제 예방
fmp_market_df = (fmp_market_df
                 .drop_duplicates(subset=['date_month_end'])
                 .sort_values('date_month_end')
                 .reset_index(drop=True))

print(f"FMP 시가총액 데이터: {len(fmp_market_df)}건")

# -----------------------------
# 안전 병합: DB가 없으면 FMP만 사용
# -----------------------------
def _safe_get_db_market_df():
    try:
        df = fetch_db_market_data(ticker, db_info)
        # None 이거나 길이 0이면 빈 DF 반환
        if df is None or len(df) == 0:
            return pd.DataFrame()
        return df.copy()
    except Exception as e:
        print(f"[WARN] DB 조회 중 예외 발생: {e}")
        return pd.DataFrame()

db_market_df = _safe_get_db_market_df()

# DB가 있으면 date_month_end 정규화 + 컬럼 정리
if not db_market_df.empty:
    # 날짜 컬럼 유도: date_month_end가 없고 date가 있으면 생성
    if 'date_month_end' not in db_market_df.columns:
        if 'date' in db_market_df.columns:
            db_market_df['date_month_end'] = db_market_df['date'].apply(convert_to_month_end)
        else:
            # 날짜 정보가 없으면 병합 불가 → 빈 DF 취급
            print("[WARN] DB 데이터에 날짜 컬럼이 없어 병합을 건너뜁니다.")
            db_market_df = pd.DataFrame()

if db_market_df.empty:
    # DB가 비어 있으면 FMP만 사용
    print("[INFO] DB 시가총액 데이터 없음 → FMP 데이터만 사용합니다.")
    merged_market_df = fmp_market_df.copy()
    # from_db 컬럼은 NaN으로 생성(분석 시 출처 구분 유용)
    merged_market_df['market_cap_billions_from_db'] = np.nan

else:
    # 필요한 컬럼명 정리
    # DB에 market_cap_billions가 있으면 rename, 없으면 NaN으로 준비
    if 'market_cap_billions' in db_market_df.columns:
        db_market_df_renamed = db_market_df.rename(
            columns={'market_cap_billions': 'market_cap_billions_from_db'}
        )
    else:
        # 필요한 최소 컬럼만 추려서 NaN 채우기
        db_market_df_renamed = db_market_df[['date_month_end']].copy()
        db_market_df_renamed['market_cap_billions_from_db'] = np.nan
        print("[WARN] DB에 'market_cap_billions' 컬럼이 없어 NaN으로 채웁니다.")

    # 병합 (분기/월말 정렬 맞춤)
    merged_market_df = fmp_market_df.merge(
        db_market_df_renamed[['date_month_end', 'market_cap_billions_from_db']],
        on='date_month_end',
        how='left'   # FMP 기준으로 맞추고 DB 값 있으면 붙임
    )

# 최종 결측 보충: FMP 값이 NaN이면 DB 값으로 대체
if 'market_cap_billions' not in merged_market_df.columns:
    # 혹시 FMP 가 다른 이름을 썼다면 여기서 보정하세요.
    # 일단 없으면 새로 만들고 DB로 채움
    merged_market_df['market_cap_billions'] = np.nan

if 'market_cap_billions_from_db' not in merged_market_df.columns:
    merged_market_df['market_cap_billions_from_db'] = np.nan

merged_market_df['market_cap_billions'] = merged_market_df['market_cap_billions'].fillna(
    merged_market_df['market_cap_billions_from_db']
)

# 정리
merged_market_df = (merged_market_df
                    .drop_duplicates(subset=['date_month_end'])
                    .sort_values('date_month_end')
                    .reset_index(drop=True))

print(f"병합 완료: {len(merged_market_df)}건 (FMP+DB)")


2. FMP 시가총액 데이터 수집 중...
FMP 시가총액 데이터: 190건
[INFO] DB 시가총액 데이터 없음 → FMP 데이터만 사용합니다.
병합 완료: 190건 (FMP+DB)


In [12]:
# merged_market_df

enhanced_merged_df = pd.merge(merged_market_df[['date_month_end', 'market_cap_billions']], rev_data, on='date_month_end', how='outer')
market_cap_resize = enhanced_merged_df[['date_month_end', 'market_cap_billions', 'ticker', 'revenue_billions']].copy()
market_cap_resize.dropna(subset =['market_cap_billions'], inplace=True)
market_cap_resize.ffill(limit=2, inplace=True)

market_cap_resize = market_cap_resize[(market_cap_resize['date_month_end'] >= start_date_month ) & (market_cap_resize['date_month_end'] <= end_date_month)]

In [13]:
market_cap_resize = market_cap_resize.dropna(axis=0)
# market_cap_resize
enhanced_merged_df_with_ttm = calculate_enhanced_ttm_and_psr(market_cap_resize)

In [14]:
from DATA.us_sarima_forecast import run_sarima_psr_only

# 12개월 예측
psr_sarima_df, psr_12_res = run_sarima_psr_only(
    df=enhanced_merged_df_with_ttm,                 # date_month_end, PSR_ttm 포함
    periods=12,                  # 12개월
    target_col="PSR_ttm",        # 다른 월간 변수로 교체 가능
    analysis_start="2012-06-01", # 2012년 이후만 분석
    warmup_months=6,             # 최초 유효값 + 6개월부터 학습
    fill_method="interpolate",   # 보간 후 ffill/bfill
    ic="aic"
)
# print(psr_12_df.tail(15)[["PSR_ttm","PSR_ttm_sarima_forecast"]])
# print(psr_12_res[ticker])
# 24개월 예측
# psr_24_df, psr_24_res = run_sarima_psr_only(rev_data, periods=24)

In [15]:
# df: 최소 ['date_month_end','PSR_ttm'] 포함, 가능하면 보조피처도 포함
psr_lstm_df, psr_results = lstm_v2.run_lstm_psr_prediction(enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12)
# psr_df에는 'PSR_ttm_lstm_forecast' 컬럼이 추가됩니다.

In [16]:
# 1) 자동 start_date (데이터 마지막 월 다음 달부터)
psr_prophet_df, psr_res = prophet_v3.run_prophet_psr_only(enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12)

15:30:33 - cmdstanpy - INFO - Chain [1] start processing


[INFO] 예측 시작일: 2025-10-31 | 데이터 마지막 월: 2025-09-30


15:30:33 - cmdstanpy - INFO - Chain [1] done processing


In [17]:
psr_es_df, psr_res_es = esmod.run_es_psr_only(
    df=enhanced_merged_df_with_ttm,  # 반드시 date_month_end / PSR_ttm 포함
    ticker= ticker,
    prediction_months=12,
    start_date=None  # None이면 자동: (데이터 max) + 1개월 말일부터
)

In [18]:
#### 4. Valuation 종합
sarima_resize_df = sarima_df[['ticker', 'revenue_billions_sarima_noexog']].copy()
lstm_resize_df = lstm_df[[ 'revenue_billions_lstm_forecast']].copy()
prophet_resize_df = prophet_raw_df[['revenue_billions_prophet_forecast']].copy()
es_resize_df = es_raw_df[['revenue_billions_esq_forecast']].copy()

revenue_forecast_df = pd.concat([sarima_resize_df, lstm_resize_df, prophet_resize_df, es_resize_df], axis=1)

psr_sarima_resiae = psr_sarima_df[['PSR_ttm_sarima_forecast']]
psr_lstm_resiae = psr_lstm_df[['PSR_ttm_lstm_forecast']]
psr_prophet_resiae = psr_prophet_df[['PSR_prophet_forecast_noexog']]
psr_es_resiae = psr_es_df[['PSR_es_forecast']]

psr_forecast_df = pd.concat([psr_sarima_resiae, psr_lstm_resiae, psr_prophet_resiae,  psr_es_resiae], axis =1)

In [19]:
revenue_forecast_ = prepare_revenue_ttm(revenue_forecast_df)
revenue_forecast_ttm = revenue_forecast_.filter(like = '_ttm')
revenue_forecast_ttm['ticker'] = ticker

In [20]:
valuation_df = pd.concat([revenue_forecast_ttm, psr_forecast_df], axis=1)

# 1) 복사본 생성 (원본 보호)
valuation_filled = valuation_df.copy()

# 2) ffill 대상 칼럼 목록 생성
cols_to_fill = ['ticker'] + [c for c in valuation_filled.columns if 'revenue' in c]

# 3) 선택된 칼럼만 ffill(limit=2)
valuation_filled[cols_to_fill] = valuation_filled[cols_to_fill].ffill(limit=2)

valuation_filled.head(5)

,revenue_billions_sarima_noexog_ttm,revenue_billions_lstm_forecast_ttm,revenue_billions_prophet_forecast_ttm,revenue_billions_esq_forecast_ttm,ticker,PSR_ttm_sarima_forecast,PSR_ttm_lstm_forecast,PSR_prophet_forecast_noexog,PSR_es_forecast
2011-05-31,2.14,2.14,2.14,2.14,MU,NaN,NaN,NaN,NaN
2011-06-30,2.14,2.14,2.14,2.14,MU,NaN,NaN,NaN,NaN
2011-07-31,2.14,2.14,2.14,2.14,MU,3.415888,3.415888,3.415888,3.415888
2011-08-31,4.28,4.28,4.28,4.28,MU,1.369159,1.369159,1.369159,1.369159
2011-09-30,4.28,4.28,4.28,4.28,MU,0.771028,0.771028,0.771028,0.771028


In [21]:
required_cols = valuation_filled.columns.tolist()

missing = [c for c in required_cols if c not in valuation_filled.columns]
if missing:
    raise ValueError(f"다음 칼럼이 없습니다: {missing}")

# 2. Valuation 계산 (revenue × PSR)
valuation_filled['sarima_valuation'] = (
    valuation_filled['revenue_billions_sarima_noexog_ttm'] *
    valuation_filled['PSR_ttm_sarima_forecast']
)

valuation_filled['lstm_valuation'] = (
    valuation_filled['revenue_billions_lstm_forecast_ttm'] *
    valuation_filled['PSR_ttm_lstm_forecast']
)

valuation_filled['prophet_valuation'] = (
    valuation_filled['revenue_billions_prophet_forecast_ttm'] *
    valuation_filled['PSR_prophet_forecast_noexog']
)

valuation_filled['es_valuation'] = (
    valuation_filled['revenue_billions_esq_forecast_ttm'] *
    valuation_filled['PSR_es_forecast']
)

# 3. 마지막 15개월 추출
# (date 칼럼이 없다면, 대신 index가 날짜인 경우로 가정)
if 'date_month_end' in valuation_filled.columns:
    valuation_filled = valuation_filled.sort_values('date_month_end')
    valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index(drop=True)
else:
    # index가 날짜라고 가정
    valuation_filled = valuation_filled.sort_index()
    valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index()


In [22]:
valuation_result

,index,revenue_billions_sarima_noexog_ttm,revenue_billions_lstm_forecast_ttm,revenue_billions_prophet_forecast_ttm,revenue_billions_esq_forecast_ttm,ticker,PSR_ttm_sarima_forecast,PSR_ttm_lstm_forecast,PSR_prophet_forecast_noexog,PSR_es_forecast,sarima_valuation,lstm_valuation,prophet_valuation,es_valuation
0,2025-07-31,33.810000,33.810000,33.810000,33.810000,MU,3.641256,3.641256,3.641256,3.641256,123.110852,123.110852,123.110852,123.110852
1,2025-08-31,37.370000,37.370000,37.370000,37.370000,MU,3.827666,3.827666,3.827666,3.827666,143.039867,143.039867,143.039867,143.039867
2,2025-09-30,37.370000,37.370000,37.370000,37.370000,MU,5.194159,5.194159,5.194159,5.194159,194.105705,194.105705,194.105705,194.105705
3,2025-10-31,37.370000,37.370000,37.370000,37.370000,MU,5.194159,3.246119,4.058253,5.204618,194.105705,121.307477,151.656929,194.496585
4,2025-11-30,40.357450,33.312922,37.773009,41.472249,MU,5.194159,3.324133,4.365488,5.215078,209.622996,110.736572,164.897613,216.281014
5,2025-12-31,40.357450,33.312922,37.773009,41.472249,MU,5.194159,3.328121,4.424042,5.225538,209.622996,110.869432,167.109397,216.714803
6,2026-01-31,40.357450,33.312922,37.773009,41.472249,MU,5.194159,3.288365,4.439693,5.235997,209.622996,109.545042,167.700556,217.148592
7,2026-02-28,43.926329,30.109856,38.694960,47.736748,MU,5.194159,3.205910,4.641754,5.246457,228.160319,96.529489,179.612494,250.448803
8,2026-03-31,43.926329,30.109856,38.694960,47.736748,MU,5.194159,3.121256,4.808984,5.256917,228.160319,93.980559,186.083435,250.948117
9,2026-04-30,43.926329,30.109856,38.694960,47.736748,MU,5.194159,3.035910,4.647302,5.267377,228.160319,91.410818,179.827148,251.447430


In [23]:
fmp_market_df.tail(10)

,ticker,date,market_cap,market_cap_billions,date_month_end
179,NVDA,2024-12-31,3288627810000,3288.63,2024-12-31
180,NVDA,2025-01-31,2934630870000,2934.63,2025-01-31
181,NVDA,2025-02-28,3053169720000,3053.17,2025-02-28
182,NVDA,2025-03-31,2648915580000,2648.92,2025-03-31
183,NVDA,2025-04-30,2662113720000,2662.11,2025-04-30
184,NVDA,2025-05-30,3302712330000,3302.71,2025-05-31
185,NVDA,2025-06-30,3861433590000,3861.43,2025-06-30
186,NVDA,2025-07-31,4347320670000,4347.32,2025-07-31
187,NVDA,2025-08-29,4244069880000,4244.07,2025-08-31
188,NVDA,2025-09-30,4546208280000,4546.21,2025-09-30


In [48]:
test = fetch_db_market_data('ANET', db_info)

In [49]:
test

,ticker,date,date_month_end,market_cap_billions
0,ANET,2000-01-31,2000-01-31,0.076800
1,ANET,2000-02-29,2000-02-29,0.143360
2,ANET,2000-03-31,2000-03-31,0.106373
3,ANET,2000-04-28,2000-04-30,0.130012
4,ANET,2000-05-31,2000-05-31,0.122132
...,...,...,...,...
150,ANET,2024-10-31,2024-10-31,2.727687
151,ANET,2024-11-29,2024-11-30,127.808954
152,ANET,2024-11-30,2024-11-30,2.864481
153,ANET,2024-12-31,2024-12-31,139.241273


In [24]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
from typing import Optional
# from stock_forecast.Korea_Market.valuation.kse_valuation_machine_v1 import psr_forecast_df


def add_repo_path():
    here = Path.cwd()
    # 현재 위치부터 상위 폴더를 훑으며 DATA 폴더가 보이는 지점 찾기
    for p in [here, *here.parents]:
        if (p / "DATA").exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return str(p)
    # 못 찾으면 로컬 고정 경로(본인 PC 경로로) 마지막 보루로 추가
    fallback = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast"
    if os.path.isdir(fallback) and fallback not in sys.path:
        sys.path.insert(0, fallback)
    return fallback

project_path = add_repo_path()
print("Using project path:", project_path)

import calendar
import time
# from DATA.stock_invest_function import *
from DATA.stock_invest_function import *
from datetime import datetime
from dateutil.relativedelta import relativedelta
warnings.filterwarnings('ignore')

# 유틸리티 함수들
def convert_to_month_end(date_str):
    try:
        # 문자열/타입 혼용 안전 변환
        date_obj = pd.to_datetime(date_str)
        if pd.isna(date_obj):
            return None

        y, m, d = date_obj.year, date_obj.month, date_obj.day

        # 1~5일 → 전달 말일
        if 1 <= d <= 5:
            if m == 1:
                prev_y, prev_m = y - 1, 12
            else:
                prev_y, prev_m = y, m - 1
            last_day_prev = calendar.monthrange(prev_y, prev_m)[1]
            return datetime(prev_y, prev_m, last_day_prev)

        # 그 외 → 해당월 말일
        last_day_cur = calendar.monthrange(y, m)[1]
        return datetime(y, m, last_day_cur)

    except Exception:
        return None

def process_daily_to_monthly_market_data(daily_data, ticker):
    if not daily_data:
        return pd.DataFrame()
    df = pd.DataFrame(daily_data)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    df['year_month'] = df['date'].dt.to_period('M')
    monthly_data = []
    for year_month in df['year_month'].unique():
        month_data = df[df['year_month'] == year_month]
        last_day_data = month_data.loc[month_data['date'].idxmax()]
        monthly_data.append({
            'ticker': ticker,
            'date': last_day_data['date'],
            'market_cap': last_day_data['marketCap'],
            'market_cap_billions': round(last_day_data['marketCap'] / 1_000_000_000, 2),
        })
    return pd.DataFrame(monthly_data)


def fetch_revenue_data(ticker, api_key):
    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
    params = {'limit': 200, 'apikey': api_key, 'period': 'quarter'}
    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code != 200:
            return None, f"HTTP {response.status_code}"
        data = response.json()
        if isinstance(data, dict) and 'Error Message' in data:
            return None, f"API 오류: {data['Error Message']}"
        if not data:
            return None, "데이터 없음"
        return data, None
    except Exception as e:
        return None, f"오류: {str(e)}"

def fetch_market_data_yearly(ticker, api_key, start_year=2010):
    all_data = []
    current_year = datetime.now().year
    for year in range(start_year, current_year + 1):
        start_date_str = f"{year}-01-01"
        end_date_str = f"{year}-12-31"
        url = f"https://financialmodelingprep.com/api/v3/historical-market-capitalization/{ticker}"
        params = {'from': start_date_str, 'to': end_date_str, 'apikey': api_key}
        try:
            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 200:
                data = response.json()
                if data and isinstance(data, list):
                    all_data.extend(data)
            time.sleep(0.3)
        except Exception as e:
            continue
    return all_data if all_data else None, None

def fetch_db_revenue_data(ticker, db_info, end_date='2025-08-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, saleq
        FROM US_fundq
        WHERE ticker = '{ticker}'
        AND saleq IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['revenue_billions'] = df['saleq'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'revenue_billions']]
    except Exception as e:
        return pd.DataFrame()

def fetch_db_market_data(ticker, db_info, end_date='2024-12-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, me
        FROM US_fundm
        WHERE ticker = '{ticker}'
        AND me IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['market_cap_billions'] = df['me'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'market_cap_billions']]
    except Exception as e:
        return pd.DataFrame()

def calculate_enhanced_ttm_and_psr(merged_data):
    """Calculate enhanced TTM and PSR"""
    df = merged_data.copy()

    # ✅ 날짜형으로 변환 (핵심 수정)
    df['date_month_end'] = pd.to_datetime(df['date_month_end'], errors='coerce')
    df = df.sort_values(['date_month_end']).reset_index(drop=True)
    df = df.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

    # Calculate TTM from quarterly revenue
    df['revenue_ttm'] = df.groupby('ticker')['revenue_billions'].rolling(window=4, min_periods=1).sum().reset_index(0,
                                                                                                                    drop=True)
    df['revenue_ttm_billions'] = df['revenue_ttm']

    # Apply 2-month shift
    df['revenue_ttm_shift'] = df.groupby('ticker')['revenue_ttm_billions'].shift(2)

    # Calculate PSR
    df['PSR_ttm'] = df['market_cap_billions'] / df['revenue_ttm_shift']

    # Handle infinite values
    df['PSR_ttm'] = df['PSR_ttm'].replace([np.inf, -np.inf], np.nan)

    return df

def prepare_revenue_ttm(
    df: pd.DataFrame,
    revenue_key: str = "revenue_billions",
    min_periods: int = 1,   # 완전한 TTM만 원하면 4로 바꾸세요
) -> pd.DataFrame:
    """
    1) revenue 칼럼들의 NaN을 '해당 행의 revenue 평균'으로 채움
    2) 각 revenue 칼럼의 4분기 합(TTM)을 *_ttm 칼럼으로 생성 (시차 없음)
    - 그룹 기준: ticker
    - 정렬 기준: date_month_end (월말 날짜)
    """
    d = df.copy()

    # --- 키 정리 ---
    # date_month_end: index에 있으면 칼럼으로 복구
    if 'date_month_end' not in d.columns:
        d = d.reset_index().rename(columns={'index': 'date_month_end'})
    d['date_month_end'] = pd.to_datetime(d['date_month_end'])

    if 'ticker' not in d.columns:
        raise ValueError("ticker 칼럼이 필요합니다.")

    # --- revenue 칼럼 자동 탐지 ---
    rev_cols = [c for c in d.columns if revenue_key in c]
    if not rev_cols:
        raise ValueError(f"'{revenue_key}' 가 포함된 칼럼을 찾지 못했습니다.")

    # --- ticker NaN 보정 ---
    # 단일 티커면 ffill/bfill로 채움, 복수 티커면 NaN 행 제거(필요 시 정책 조정)
    uniq_tickers = d['ticker'].dropna().unique()
    if len(uniq_tickers) == 1:
        d['ticker'] = d['ticker'].ffill().bfill()
    else:
        d = d[~d['ticker'].isna()].copy()

    # --- 정렬 ---
    d = d.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

    # --- NaN 보간: 행 단위 평균으로 revenue 결측치 채우기 ---
    row_mean = d[rev_cols].mean(axis=1, skipna=True)
    for c in rev_cols:
        d[c] = d[c].fillna(row_mean)

    # --- TTM 계산 (최근 4분기 합, 시차 없음) ---
    for c in rev_cols:
        ttm_col = f"{c}_ttm"
        d[ttm_col] = (
            d.groupby('ticker', group_keys=False)[c]
             .rolling(window=4, min_periods=min_periods)
             .sum()
             .reset_index(level=0, drop=True)
        )

    d = d.set_index('date_month_end')

    return d

def clean_rev_data(rev_data: pd.DataFrame) -> pd.DataFrame:
    """
    1) 'revenue' 컬럼 값이 NaN인 행 제거
    2) (calendar_year, period) 중복 행 제거 (첫 번째 행만 유지)
       - 입력 순서를 그대로 기준으로 '첫째 데이터'를 보존
    """
    required = ['revenue', 'calendar_year', 'period']
    missing = [c for c in required if c not in rev_data.columns]
    if missing:
        raise ValueError(f"필수 컬럼이 없습니다: {missing}")

    d = rev_data.copy()

    # 1) revenue NaN인 행 제거
    before = len(d)
    d = d[~d['revenue'].isna()].copy()
    removed_nan = before - len(d)

    # 2) (calendar_year, period) 중복 제거 — 첫 행 유지(현재 순서 기준)
    before2 = len(d)
    d = d.drop_duplicates(subset=['calendar_year', 'period'], keep='first').reset_index(drop=True)
    removed_dup = before2 - len(d)

    print(f"[clean_rev_data_minimal] removed rows → revenue NaN: {removed_nan}, duplicates: {removed_dup}")
    return d

# ==========================
# 사용 예시
# ==========================
# cleaned = clean_rev_data(rev_data)
# cleaned.head()

# 설정값들
ticker = 'MU'

hs_code = '841191'

api_key = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    # 'host': '192.168.0.230',
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

start_date_month = '2011-03-01'
end_date_month = (pd.Timestamp.today().normalize() - pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')

print("=" * 80)
print("전처리 과정 테스트 시작")
print(f"대상 종목: {ticker}")
print("=" * 80)

# 1. FMP 매출 데이터 수집
print("\n1. FMP 매출 데이터 수집 중...")
revenue_data, error = fetch_revenue_data(ticker, api_key)

if revenue_data is None:
    print(f"ERROR: FMP 매출 데이터 수집 실패 - {error}")
    exit()

all_revenue_data = []
for item in revenue_data:
    all_revenue_data.append({
        'ticker': ticker,
        'date': item.get('date', ''),
        'calendar_year': item.get('calendarYear', ''),
        'period': item.get('period', ''),
        'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
        'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
    })

fmp_revenue_df = pd.DataFrame(all_revenue_data)
fmp_revenue_df['date'] = pd.to_datetime(fmp_revenue_df['date'])
fmp_revenue_df = fmp_revenue_df.sort_values(['ticker', 'date'])
fmp_revenue_df['date_month_end'] = fmp_revenue_df['date'].apply(convert_to_month_end)
fmp_revenue_df = fmp_revenue_df.drop_duplicates(subset=['date_month_end'], keep='first').reset_index(drop=True)
print(f"FMP 매출 데이터: {len(fmp_revenue_df)}건")


# 2. DB 매출 데이터 가져오기
db_revenue_raw = fetch_db_revenue_data(ticker, db_info)
db_revenue_df = db_revenue_raw.loc[db_revenue_raw['revenue_billions'] != db_revenue_raw['revenue_billions'].shift()]

# db_revenue_df = db_revenue_raw.drop_duplicates(subset=['date_month_end', 'revenue_billions'], keep='first')
mereged_rev_data = pd.merge(fmp_revenue_df, db_revenue_df, on = ['ticker', 'date_month_end'], how='outer')

rev_data = mereged_rev_data[mereged_rev_data['date_month_end'] >= start_date_month ]
rev_data['revenue_billions_x'] = rev_data['revenue_billions_x'].fillna(rev_data['revenue_billions_y'])
# 컬럼 이름 변경
rev_data.rename(columns={'revenue_billions_x': 'revenue_billions'}, inplace=True)

rev_data = clean_rev_data(rev_data)

import importlib
import DATA.us_sarima_forecast as sarima
importlib.reload(sarima)
import DATA.us_lstm_forecast_v2 as lstm_v2
importlib.reload(lstm_v2)
import DATA.us_prophet_forecast_v3 as prophet_v3
importlib.reload(prophet_v3)
import DATA.us_est_forecast_v2 as esmod
importlib.reload(esmod)

# periods=4 또는 8 등 원하는 분기 수
periods = 4

# 모듈 함수 호출 (내부에서 월말 정렬/중복제거 처리)
sarima_df, results = sarima.run_sarima_prediction(
    rev_data,
    forecast_quarters=periods,   # ← 예측 분기 수
    exog_col=None                # 외생변수 없으면 None
)

# 인덱스를 date_month_end로 설정
sarima_df = sarima_df.sort_values("date_month_end").set_index("date_month_end")
# 1) 4분기 예측
# 4분기 예측
lstm_raw_df, lstm_results_4q = lstm_v2.run_lstm_revenue_prediction(rev_data, ticker=ticker, prediction_quarters=4)
lstm_df = lstm_raw_df.drop_duplicates(subset=['revenue_billions_lstm_forecast'], keep='last')
# 4분기 예측
prophet_raw_df, res_4q = prophet_v3.run_prophet_revenue_only(rev_data, ticker=ticker, prediction_quarters=4)
# 4분기 예측
es_raw_df, res_q4 = esmod.run_es_revenue_quarterly(rev_data, ticker=ticker, prediction_quarters=4)

# 3. FMP 시가총액 데이터 수집
print("2. FMP 시가총액 데이터 수집 중...")
market_data, error = fetch_market_data_yearly(ticker, api_key, start_year=2010)

if not market_data:
    print("ERROR: FMP 시가총액 데이터 수집 실패")
    raise SystemExit(1)

fmp_market_df = process_daily_to_monthly_market_data(market_data, ticker).copy()
fmp_market_df['date_month_end'] = fmp_market_df['date'].apply(convert_to_month_end)
# 혹시 중복/정렬 문제 예방
fmp_market_df = (fmp_market_df
                 .drop_duplicates(subset=['date_month_end'])
                 .sort_values('date_month_end')
                 .reset_index(drop=True))

print(f"FMP 시가총액 데이터: {len(fmp_market_df)}건")

# -----------------------------
# 안전 병합: DB가 없으면 FMP만 사용
# -----------------------------
def _safe_get_db_market_df():
    try:
        df = fetch_db_market_data(ticker, db_info)
        # None 이거나 길이 0이면 빈 DF 반환
        if df is None or len(df) == 0:
            return pd.DataFrame()
        return df.copy()
    except Exception as e:
        print(f"[WARN] DB 조회 중 예외 발생: {e}")
        return pd.DataFrame()

db_market_df = _safe_get_db_market_df()

# DB가 있으면 date_month_end 정규화 + 컬럼 정리
if not db_market_df.empty:
    # 날짜 컬럼 유도: date_month_end가 없고 date가 있으면 생성
    if 'date_month_end' not in db_market_df.columns:
        if 'date' in db_market_df.columns:
            db_market_df['date_month_end'] = db_market_df['date'].apply(convert_to_month_end)
        else:
            # 날짜 정보가 없으면 병합 불가 → 빈 DF 취급
            print("[WARN] DB 데이터에 날짜 컬럼이 없어 병합을 건너뜁니다.")
            db_market_df = pd.DataFrame()

if db_market_df.empty:
    # DB가 비어 있으면 FMP만 사용
    print("[INFO] DB 시가총액 데이터 없음 → FMP 데이터만 사용합니다.")
    merged_market_df = fmp_market_df.copy()
    # from_db 컬럼은 NaN으로 생성(분석 시 출처 구분 유용)
    merged_market_df['market_cap_billions_from_db'] = np.nan

else:
    # 필요한 컬럼명 정리
    # DB에 market_cap_billions가 있으면 rename, 없으면 NaN으로 준비
    if 'market_cap_billions' in db_market_df.columns:
        db_market_df_renamed = db_market_df.rename(
            columns={'market_cap_billions': 'market_cap_billions_from_db'}
        )
    else:
        # 필요한 최소 컬럼만 추려서 NaN 채우기
        db_market_df_renamed = db_market_df[['date_month_end']].copy()
        db_market_df_renamed['market_cap_billions_from_db'] = np.nan
        print("[WARN] DB에 'market_cap_billions' 컬럼이 없어 NaN으로 채웁니다.")

    # 병합 (분기/월말 정렬 맞춤)
    merged_market_df = fmp_market_df.merge(
        db_market_df_renamed[['date_month_end', 'market_cap_billions_from_db']],
        on='date_month_end',
        how='left'   # FMP 기준으로 맞추고 DB 값 있으면 붙임
    )

# 최종 결측 보충: FMP 값이 NaN이면 DB 값으로 대체
if 'market_cap_billions' not in merged_market_df.columns:
    # 혹시 FMP 가 다른 이름을 썼다면 여기서 보정하세요.
    # 일단 없으면 새로 만들고 DB로 채움
    merged_market_df['market_cap_billions'] = np.nan

if 'market_cap_billions_from_db' not in merged_market_df.columns:
    merged_market_df['market_cap_billions_from_db'] = np.nan

merged_market_df['market_cap_billions'] = merged_market_df['market_cap_billions'].fillna(
    merged_market_df['market_cap_billions_from_db']
)

# 정리
merged_market_df = (merged_market_df
                    .drop_duplicates(subset=['date_month_end'])
                    .sort_values('date_month_end')
                    .reset_index(drop=True))

print(f"병합 완료: {len(merged_market_df)}건 (FMP+DB)")


# merged_market_df

enhanced_merged_df = pd.merge(merged_market_df[['date_month_end', 'market_cap_billions']], rev_data, on='date_month_end', how='outer')
market_cap_resize = enhanced_merged_df[['date_month_end', 'market_cap_billions', 'ticker', 'revenue_billions']].copy()
market_cap_resize.dropna(subset =['market_cap_billions'], inplace=True)
market_cap_resize.ffill(limit=2, inplace=True)

market_cap_resize = market_cap_resize[(market_cap_resize['date_month_end'] >= start_date_month ) & (market_cap_resize['date_month_end'] <= end_date_month)]

market_cap_resize = market_cap_resize.dropna(axis=0)
# market_cap_resize
enhanced_merged_df_with_ttm = calculate_enhanced_ttm_and_psr(market_cap_resize)

from DATA.us_sarima_forecast import run_sarima_psr_only

# 12개월 예측
psr_sarima_df, psr_12_res = run_sarima_psr_only(
    df=enhanced_merged_df_with_ttm,                 # date_month_end, PSR_ttm 포함
    periods=12,                  # 12개월
    target_col="PSR_ttm",        # 다른 월간 변수로 교체 가능
    analysis_start="2012-06-01", # 2012년 이후만 분석
    warmup_months=6,             # 최초 유효값 + 6개월부터 학습
    fill_method="interpolate",   # 보간 후 ffill/bfill
    ic="aic"
)
# print(psr_12_df.tail(15)[["PSR_ttm","PSR_ttm_sarima_forecast"]])
# print(psr_12_res[ticker])
# 24개월 예측
# psr_24_df, psr_24_res = run_sarima_psr_only(rev_data, periods=24)

# df: 최소 ['date_month_end','PSR_ttm'] 포함, 가능하면 보조피처도 포함
psr_lstm_df, psr_results = lstm_v2.run_lstm_psr_prediction(enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12)
# 1) 자동 start_date (데이터 마지막 월 다음 달부터)
psr_prophet_df, psr_res = prophet_v3.run_prophet_psr_only(enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12)
psr_es_df, psr_res_es = esmod.run_es_psr_only(
    df=enhanced_merged_df_with_ttm,  # 반드시 date_month_end / PSR_ttm 포함
    ticker= ticker,
    prediction_months=12,
    start_date=None  # None이면 자동: (데이터 max) + 1개월 말일부터
)

#### 4. Valuation 종합
sarima_resize_df = sarima_df[['ticker', 'revenue_billions_sarima_noexog']].copy()
lstm_resize_df = lstm_df[[ 'revenue_billions_lstm_forecast']].copy()
prophet_resize_df = prophet_raw_df[['revenue_billions_prophet_forecast']].copy()
es_resize_df = es_raw_df[['revenue_billions_esq_forecast']].copy()

revenue_forecast_df = pd.concat([sarima_resize_df, lstm_resize_df, prophet_resize_df, es_resize_df], axis=1)

psr_sarima_resiae = psr_sarima_df[['PSR_ttm_sarima_forecast']]
psr_lstm_resiae = psr_lstm_df[['PSR_ttm_lstm_forecast']]
psr_prophet_resiae = psr_prophet_df[['PSR_prophet_forecast_noexog']]
psr_es_resiae = psr_es_df[['PSR_es_forecast']]

psr_forecast_df = pd.concat([psr_sarima_resiae, psr_lstm_resiae, psr_prophet_resiae,  psr_es_resiae], axis =1)

revenue_forecast_ = prepare_revenue_ttm(revenue_forecast_df)
revenue_forecast_ttm = revenue_forecast_.filter(like = '_ttm')
revenue_forecast_ttm['ticker'] = ticker

valuation_df = pd.concat([revenue_forecast_ttm, psr_forecast_df], axis=1)

# 1) 복사본 생성 (원본 보호)
valuation_filled = valuation_df.copy()

# 2) ffill 대상 칼럼 목록 생성
cols_to_fill = ['ticker'] + [c for c in valuation_filled.columns if 'revenue' in c]

# 3) 선택된 칼럼만 ffill(limit=2)
valuation_filled[cols_to_fill] = valuation_filled[cols_to_fill].ffill(limit=2)

required_cols = valuation_filled.columns.tolist()

missing = [c for c in required_cols if c not in valuation_filled.columns]
if missing:
    raise ValueError(f"다음 칼럼이 없습니다: {missing}")

# 2. Valuation 계산 (revenue × PSR)
valuation_filled['sarima_valuation'] = (
    valuation_filled['revenue_billions_sarima_noexog_ttm'] *
    valuation_filled['PSR_ttm_sarima_forecast']
)

valuation_filled['lstm_valuation'] = (
    valuation_filled['revenue_billions_lstm_forecast_ttm'] *
    valuation_filled['PSR_ttm_lstm_forecast']
)

valuation_filled['prophet_valuation'] = (
    valuation_filled['revenue_billions_prophet_forecast_ttm'] *
    valuation_filled['PSR_prophet_forecast_noexog']
)

valuation_filled['es_valuation'] = (
    valuation_filled['revenue_billions_esq_forecast_ttm'] *
    valuation_filled['PSR_es_forecast']
)

# 3. 마지막 15개월 추출
# (date 칼럼이 없다면, 대신 index가 날짜인 경우로 가정)
if 'date_month_end' in valuation_filled.columns:
    valuation_filled = valuation_filled.sort_values('date_month_end')
    valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index(drop=True)
else:
    # index가 날짜라고 가정
    valuation_filled = valuation_filled.sort_index()
    valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index()







Using project path: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
전처리 과정 테스트 시작
대상 종목: MU

1. FMP 매출 데이터 수집 중...
FMP 매출 데이터: 161건
[clean_rev_data_minimal] removed rows → revenue NaN: 0, duplicates: 0


15:49:53 - cmdstanpy - INFO - Chain [1] start processing
15:49:53 - cmdstanpy - INFO - Chain [1] done processing


2. FMP 시가총액 데이터 수집 중...
FMP 시가총액 데이터: 190건
[INFO] DB 시가총액 데이터 없음 → FMP 데이터만 사용합니다.
병합 완료: 190건 (FMP+DB)


15:50:40 - cmdstanpy - INFO - Chain [1] start processing
15:50:40 - cmdstanpy - INFO - Chain [1] done processing


[INFO] 예측 시작일: 2025-10-31 | 데이터 마지막 월: 2025-09-30


In [25]:
valuation_result

,index,revenue_billions_sarima_noexog_ttm,revenue_billions_lstm_forecast_ttm,revenue_billions_prophet_forecast_ttm,revenue_billions_esq_forecast_ttm,ticker,PSR_ttm_sarima_forecast,PSR_ttm_lstm_forecast,PSR_prophet_forecast_noexog,PSR_es_forecast,sarima_valuation,lstm_valuation,prophet_valuation,es_valuation
0,2025-07-31,33.810000,33.810000,33.810000,33.810000,MU,3.641256,3.641256,3.641256,3.641256,123.110852,123.110852,123.110852,123.110852
1,2025-08-31,37.370000,37.370000,37.370000,37.370000,MU,3.827666,3.827666,3.827666,3.827666,143.039867,143.039867,143.039867,143.039867
2,2025-09-30,37.370000,37.370000,37.370000,37.370000,MU,5.194159,5.194159,5.194159,5.194159,194.105705,194.105705,194.105705,194.105705
3,2025-10-31,37.370000,37.370000,37.370000,37.370000,MU,5.194159,3.166947,4.058253,5.204618,194.105705,118.348814,151.656929,194.496585
4,2025-11-30,40.357450,32.966814,37.773009,41.472249,MU,5.194159,3.263127,4.365488,5.215078,209.622996,107.574911,164.897613,216.281014
5,2025-12-31,40.357450,32.966814,37.773009,41.472249,MU,5.194159,3.295069,4.424042,5.225538,209.622996,108.627910,167.109397,216.714803
6,2026-01-31,40.357450,32.966814,37.773009,41.472249,MU,5.194159,3.281728,4.439693,5.235997,209.622996,108.188109,167.700556,217.148592
7,2026-02-28,43.926329,29.267891,38.694960,47.736748,MU,5.194159,3.212942,4.641754,5.246457,228.160319,94.036048,179.612494,250.448803
8,2026-03-31,43.926329,29.267891,38.694960,47.736748,MU,5.194159,3.136056,4.808984,5.256917,228.160319,91.785737,186.083435,250.948117
9,2026-04-30,43.926329,29.267891,38.694960,47.736748,MU,5.194159,3.052892,4.647302,5.267377,228.160319,89.351717,179.827148,251.447430
